# LangChain Guardrails — Masking & Safety

### What we'll cover:
```
Part 1 → Regex-based Email Masking
Part 2 → PII Masking (Email + Phone using presidio)
Part 3 → LangChain Guardrails (input + output safety checks)
```

In [2]:
# Install all required libraries
!pip install langchain langchain-openai langchain-experimental presidio-analyzer presidio-anonymizer faker

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp312-cp312-win_amd64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp312-cp312-win_amd64.whl.metadata (9.9 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached blis-1.3.3-cp312-cp312-win_amd64.whl.metadata (7.7 kB)
  Using cached cloudpathlib-0.23.0-py3-none-any.whl.metadata (16 kB)
  Using cached requests_file-3.0.1-py2.py3-none-any.whl.metadata (1.7 kB)
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.6 MB 4.2 MB/s eta 0:00:01
   ------------------------ --------------- 1.6/2.6 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 2.4/2.6 MB 4.5 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 3.8 MB/s  0:00:

In [3]:
!pip install spacy

In [5]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 1.3 MB/s eta 0:00:09
     ---- ----------------------------------- 1.3/12.8 MB 1.8 MB/s eta 0:00:07
     ---- ----------------------------------- 1.6/12.8 MB 1.8 MB/s eta 0:00:07
     ----- ---------------------------------- 1.8/12.8 MB 1.8 MB/s eta 0:00:07
     ----- ---------------------------------- 1.8/12.8 MB 1.8 MB/s eta 0:00:07
     ------- -------------------------------- 2.4/12.8 MB 1.4 MB/s eta 0:00:08
     -------- ------------------------------- 2.6/12.8 MB 1.4 MB/s eta 0:00:08
     -------- ------------------------------- 2.6/12.8 MB 1.4 MB/s eta 0:00:08
     --------- ------------------------------ 3.1/12.8 MB 1.4 MB/s eta 0:00:08
 

In [6]:
from dotenv import load_dotenv
import httpx

load_dotenv()

True

---
# PART 1 — Regex-Based Email Masking

**What is it?**  
Before sending text to an LLM, we scan it with a regular expression pattern 
and replace any email addresses with a placeholder like `[EMAIL MASKED]`.

**Why?**  
Simple, fast, zero dependencies — good for well-structured data like emails/IDs.

In [7]:
import re

# The regex pattern that matches email addresses
# Breaking it down:
#   [a-zA-Z0-9._%+-]+   → username (letters, digits, dots, underscores etc.)
#   @                   → literal @ symbol
#   [a-zA-Z0-9.-]+      → domain name (e.g. gmail, company)
#   \.                  → literal dot
#   [a-zA-Z]{2,}        → TLD (com, org, in, io etc.) — at least 2 chars

EMAIL_PATTERN = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

def mask_emails(text: str, placeholder: str = "[EMAIL MASKED]") -> str:
    """
    Finds all email addresses in the text and replaces them with placeholder.
    """
    found = re.findall(EMAIL_PATTERN, text)
    masked = re.sub(EMAIL_PATTERN, placeholder, text)
    return masked, found


# --- Test it ---
sample_text = """
Hi, my name is John. Please contact me at john.doe@gmail.com
or reach my manager at manager_hr@company.org for further assistance.
My backup email is john123@yahoo.co.in
"""

masked_text, found_emails = mask_emails(sample_text)

print("ORIGINAL TEXT:")
print(sample_text)
print("="*60)
print("EMAILS FOUND:", found_emails)
print("="*60)
print("MASKED TEXT:")
print(masked_text)

ORIGINAL TEXT:

Hi, my name is John. Please contact me at john.doe@gmail.com
or reach my manager at manager_hr@company.org for further assistance.
My backup email is john123@yahoo.co.in

EMAILS FOUND: ['john.doe@gmail.com', 'manager_hr@company.org', 'john123@yahoo.co.in']
MASKED TEXT:

Hi, my name is John. Please contact me at [EMAIL MASKED]
or reach my manager at [EMAIL MASKED] for further assistance.
My backup email is [EMAIL MASKED]



In [9]:
# --- Now plug it into an LLM call ---
# The LLM never sees the real email addresses

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    http_client=httpx.Client(verify=False)
)

def ask_with_email_masking(user_input: str) -> str:
    # Step 1: mask emails BEFORE sending to LLM
    safe_input, found = mask_emails(user_input)

    print(f"📧 Emails detected & masked: {found}")
    print(f"📤 Sending to LLM (masked): {safe_input.strip()}")
    print("-"*60)

    # Step 2: send the masked text to LLM
    response = llm.invoke([HumanMessage(content=safe_input)])
    return response.content


user_input = "Summarize this: Please email sarah.jones@company.com for the Q3 report details."

answer = ask_with_email_masking(user_input)
print("🤖 LLM RESPONSE:")
print(answer)

📧 Emails detected & masked: ['sarah.jones@company.com']
📤 Sending to LLM (masked): Summarize this: Please email [EMAIL MASKED] for the Q3 report details.
------------------------------------------------------------
🤖 LLM RESPONSE:
For details on the Q3 report, please email [EMAIL MASKED].


---
# PART 2 — PII Masking (Email + Phone)

**What is PII?**  
Personally Identifiable Information — any data that could identify a person:  
emails, phone numbers, names, addresses, credit card numbers, etc.

**Why not just use regex for everything?**  
Phone numbers have dozens of formats: `+91-9876543210`, `(123) 456-7890`, `123.456.7890`  
Regex gets messy fast. **Microsoft Presidio** is built exactly for this.

In [10]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

# Presidio has two components:
# AnalyzerEngine  → DETECTS PII (finds where it is)
# AnonymizerEngine → MASKS PII  (replaces it)

analyzer  = AnalyzerEngine()
anonymizer = AnonymizerEngine()

print("✅ Presidio engines ready")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Presidio engines ready


In [12]:
# --- Step 1: Just DETECT PII first ---

sample_pii_text = """
Hello, I am Ravi Kumar. You can reach me at ravi.kumar@example.com
or call me on +91-9876543210. My office number is (022) 4567-8901.
Also feel free to email my colleague at priya@startup.io
"""

# Analyze — detect all PII entities
results = analyzer.analyze(
    text=sample_pii_text,
    entities=["EMAIL_ADDRESS", "PHONE_NUMBER"],  # what to look for
    language="en"
)

print("PII DETECTED:")
print("-"*60)
for r in results:
    detected_value = sample_pii_text[r.start:r.end]
    print(f"Type: {r.entity_type:<20} | Value: {detected_value:<30} | Confidence: {r.score:.2f}")

PII DETECTED:
------------------------------------------------------------
Type: EMAIL_ADDRESS        | Value: ravi.kumar@example.com         | Confidence: 1.00
Type: EMAIL_ADDRESS        | Value: priya@startup.io               | Confidence: 1.00
Type: PHONE_NUMBER         | Value: (022) 4567-8901                | Confidence: 0.75
Type: PHONE_NUMBER         | Value: +91-9876543210                 | Confidence: 0.40


In [13]:
# --- Step 2: MASK the detected PII ---

# Custom placeholders for each PII type
operators = {
    "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL MASKED]"}),
    "PHONE_NUMBER":  OperatorConfig("replace", {"new_value": "[PHONE MASKED]"}),
}

anonymized = anonymizer.anonymize(
    text=sample_pii_text,
    analyzer_results=results,
    operators=operators
)

print("ORIGINAL:")
print(sample_pii_text)
print("="*60)
print("AFTER PII MASKING:")
print(anonymized.text)

ORIGINAL:

Hello, I am Ravi Kumar. You can reach me at ravi.kumar@example.com
or call me on +91-9876543210. My office number is (022) 4567-8901.
Also feel free to email my colleague at priya@startup.io

AFTER PII MASKING:

Hello, I am Ravi Kumar. You can reach me at [EMAIL MASKED]
or call me on [PHONE MASKED]. My office number is [PHONE MASKED].
Also feel free to email my colleague at [EMAIL MASKED]



In [14]:
# --- Reusable PII masking function ---

def mask_pii(text: str, entities: list = ["EMAIL_ADDRESS", "PHONE_NUMBER"]) -> str:
    """
    Detects and masks PII in the given text.
    Returns the masked text.
    """
    analysis = analyzer.analyze(text=text, entities=entities, language="en")

    if not analysis:
        print("ℹ️ No PII detected.")
        return text

    print(f"🔐 PII found: {[f'{r.entity_type}={text[r.start:r.end]}' for r in analysis]}")

    operators = {
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL MASKED]"}),
        "PHONE_NUMBER":  OperatorConfig("replace", {"new_value": "[PHONE MASKED]"}),
    }

    result = anonymizer.anonymize(text=text, analyzer_results=analysis, operators=operators)
    return result.text


# --- Plug PII masking into LLM call ---

def ask_with_pii_masking(user_input: str) -> str:
    # Mask BEFORE sending to LLM
    safe_input = mask_pii(user_input)
    print(f"📤 Sending to LLM: {safe_input.strip()}")
    print("-"*60)
    response = llm.invoke([HumanMessage(content=safe_input)])
    return response.content


user_input = "Can you write a formal email invite for Priya at priya@company.com, phone +91-9123456789?"
answer = ask_with_pii_masking(user_input)
print("🤖 LLM RESPONSE:")
print(answer)

🔐 PII found: ['EMAIL_ADDRESS=priya@company.com', 'PHONE_NUMBER=+91-9123456789']
📤 Sending to LLM: Can you write a formal email invite for Priya at [EMAIL MASKED], phone [PHONE MASKED]?
------------------------------------------------------------
🤖 LLM RESPONSE:
Subject: Invitation to [Event/Meeting Name]

Dear Priya,

I hope this message finds you well.

I am writing to formally invite you to [briefly describe the event or meeting, e.g., "our upcoming team meeting" or "the project kickoff meeting"]. The details are as follows:

**Date:** [Insert date]  
**Time:** [Insert time]  
**Location:** [Insert location or specify if it will be a virtual meeting with a link]  
**Agenda:** [Briefly outline the agenda or purpose of the meeting]

Your insights and contributions would be greatly valued, and we would be delighted to have you join us.

Please let me know if you will be able to attend. If you have any questions or need further information, feel free to reach out to me at [Your Email] or

---
# PART 3 — LangChain Guardrails

**What are guardrails?**  
Rules that sit *around* the LLM to control what goes IN and what comes OUT.

```
User Input
    ↓
[INPUT GUARDRAIL]  ← block/mask bad input before LLM sees it
    ↓
   LLM
    ↓
[OUTPUT GUARDRAIL] ← check/block bad output before user sees it
    ↓
Final Response
```

We'll build this using **LangChain's `RunnableLambda`** to wrap checks around the chain.

In [15]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

# --- Define our guardrail checks ---

# Blocked topics — input guardrail
BLOCKED_KEYWORDS = ["hack", "exploit", "malware", "bomb", "illegal", "weapon"]

def input_guardrail(input_dict: dict) -> dict:
    """
    INPUT GUARDRAIL:
    1. Check for blocked/harmful keywords
    2. Mask any PII found in the input
    """
    user_message = input_dict["input"]

    # Check 1: blocked keywords
    lower = user_message.lower()
    for keyword in BLOCKED_KEYWORDS:
        if keyword in lower:
            raise ValueError(f"🚫 INPUT BLOCKED: Contains forbidden topic → '{keyword}'")

    # Check 2: mask PII before sending to LLM
    safe_message = mask_pii(user_message)

    print(f"✅ Input guardrail passed")
    return {"input": safe_message}


def output_guardrail(output: str) -> str:
    """
    OUTPUT GUARDRAIL:
    1. Check if LLM response contains any PII it shouldn't
    2. Check if response contains any blocked content
    """
    # Check 1: scan output for any leaked PII
    analysis = analyzer.analyze(
        text=output,
        entities=["EMAIL_ADDRESS", "PHONE_NUMBER"],
        language="en"
    )
    if analysis:
        print("⚠️  PII found in LLM output — masking it...")
        output = mask_pii(output)

    # Check 2: block harmful output
    lower = output.lower()
    for keyword in BLOCKED_KEYWORDS:
        if keyword in lower:
            return "🚫 OUTPUT BLOCKED: Response contained unsafe content."

    print("✅ Output guardrail passed")
    return output


print("Guardrail functions defined ✅")

Guardrail functions defined ✅


In [16]:
# --- Build the Guardrailed Chain ---

# Prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer clearly and concisely."),
    ("human",  "{input}")
])

# The core LLM chain
llm_chain = prompt | llm

# Wrap with guardrails using RunnableLambda
guardrailed_chain = (
    RunnableLambda(input_guardrail)   # 1. Check & clean input
    | llm_chain                        # 2. Run LLM
    | RunnableLambda(lambda r: output_guardrail(r.content))  # 3. Check output
)

print("Guardrailed chain ready ✅")

Guardrailed chain ready ✅


In [17]:
# --- TEST 1: Normal safe query ---
print("TEST 1: Normal query")
print("="*60)

response = guardrailed_chain.invoke({"input": "What is the capital of France?"})
print(f"\n🤖 Response: {response}")

TEST 1: Normal query
ℹ️ No PII detected.
✅ Input guardrail passed
✅ Output guardrail passed

🤖 Response: The capital of France is Paris.


In [18]:
# --- TEST 2: Query with PII — gets masked automatically ---
print("TEST 2: Query with PII (email + phone)")
print("="*60)

response = guardrailed_chain.invoke({
    "input": "Draft a message to john.smith@company.com who called from +91-9988776655 about a meeting."
})
print(f"\n🤖 Response: {response}")

TEST 2: Query with PII (email + phone)
🔐 PII found: ['EMAIL_ADDRESS=john.smith@company.com', 'PHONE_NUMBER=+91-9988776655']
✅ Input guardrail passed
✅ Output guardrail passed

🤖 Response: Subject: Follow-Up on Our Recent Call

Dear [Recipient's Name],

I hope this message finds you well. I wanted to follow up regarding our recent call about the meeting. Please let me know your availability so we can finalize the details.

Looking forward to hearing from you soon.

Best regards,

[Your Name]  
[Your Position]  
[Your Contact Information]  
[Your Company]  


In [19]:
# --- TEST 3: Blocked input — harmful keyword ---
print("TEST 3: Harmful input (blocked keyword)")
print("="*60)

try:
    response = guardrailed_chain.invoke({
        "input": "How do I hack into a system?"
    })
    print(f"Response: {response}")
except ValueError as e:
    print(e)

TEST 3: Harmful input (blocked keyword)
🚫 INPUT BLOCKED: Contains forbidden topic → 'hack'


---
## Summary

```
PART 1 — Regex Masking
  Simple pattern match → replace emails with [EMAIL MASKED]
  Best for: structured, predictable formats

PART 2 — PII Masking with Presidio
  AnalyzerEngine  → detects PII (email, phone, name, etc.)
  AnonymizerEngine → replaces with custom placeholders
  Best for: real-world messy text with varied formats

PART 3 — LangChain Guardrails
  Input Guardrail  → blocks harmful topics + masks PII before LLM
  Output Guardrail → scans LLM response for leaked PII or unsafe content
  Built with RunnableLambda wrapping the chain
```

| Approach | What it catches | Complexity |
|---|---|---|
| Regex | Exact-format emails | Low |
| Presidio PII | Emails, phones, names, IDs | Medium |
| Guardrails | Topics + PII + output safety | High |